# 05 — 训练产物验收(单 run 体检 + xc 多模型横向对比)

对应 runbook 第 4 步检查点:manifest 口径快照完整、OOT/Valid 不大幅劣于 Train、calibration.json 健康(否则该 bundle 不能参与校准融合)、头部特征业务可解释。改 `PRODUCT` / `RUN_ID` 看任意 run;最后一格横向汇总 5 个 xc 模型同一 run-id 的指标。

In [ ]:
import sys
sys.path.insert(0, '../src')
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display
from wdm.config import load_config
from wdm.utils.paths import model_run_dir

PRODUCT = 'xc_resp_finish'
RUN_ID = 'prod_20260610'

cfg = load_config(PRODUCT)
rd = model_run_dir(cfg, RUN_ID)
print('run dir:', rd)
man_path = rd / 'run_manifest.json'
if man_path.is_file():
    manifest = json.loads(man_path.read_text(encoding='utf-8'))
    for k in ['run_id', 'product', 'selected_features_version', 'label_column', 'n_features_total',
              'tuner_objective', 'cv_strategy', 'sample_weight', 'split_strategy', 'split_ratios']:
        print('{0}: {1}'.format(k, manifest.get(k)))
    print('split_boundaries:', json.dumps(manifest.get('split_boundaries'), ensure_ascii=False))
else:
    print('警告:无 run_manifest.json(旧 bundle?)')

In [ ]:
# 全量指标 + 退化检查(oot/train 腰斩 = 过拟合/时间漂移嫌疑,runbook 第 4 步)
print(open(rd / 'metrics.md', encoding='utf-8').read())
m = pd.read_json(str(rd / 'metrics.json')).set_index('split')
cols = ['pr_auc', 'ks', 'lift_at_k', 'precision_at_k']
display(m[['n', 'base_rate'] + cols].round(4))
display((m.loc['oot', cols] / m.loc['train', cols]).rename('oot/train').round(3))

In [ ]:
# calibration.json 体检:x 严格单调递增、y 单调不减且在 [0,1]。
# 缺失/异常 → 该 bundle 不能参与校准融合(funnel eval 会回退原始分并告警,生产验收不允许)
cal_path = rd / 'calibration.json'
if cal_path.is_file():
    cal = json.loads(cal_path.read_text(encoding='utf-8'))
    x, y = np.asarray(cal['x'], dtype=float), np.asarray(cal['y'], dtype=float)
    print('points:', len(x),
          '| x 严格递增:', bool(np.all(np.diff(x) > 0)),
          '| y 单调不减:', bool(np.all(np.diff(y) >= 0)),
          '| y in [0,1]:', bool((y >= 0).all() and (y <= 1).all()),
          '| n_fit:', cal.get('n_fit'), '| n_pos:', cal.get('n_pos'))
    plt.figure(figsize=(5, 4))
    plt.plot(x, y, marker='.', ms=3)
    plt.plot([0, 1], [0, 1], ls='--', lw=0.8, color='gray')
    plt.xlabel('raw score'); plt.ylabel('calibrated')
    plt.title('isotonic calibration map')
    plt.grid(alpha=0.3); plt.tight_layout()
else:
    print('警告:无 calibration.json(valid 样本/正例过少被跳过?)— 排查 valid 窗口;'
          '该 bundle 不能参与校准融合')

In [ ]:
# 特征重要性 + 各切分分位累计 lift
display(pd.read_csv(rd / 'importance.csv').head(20))
fig, ax = plt.subplots(figsize=(7, 4))
for split in ['train', 'valid', 'oot']:
    p = rd / 'binned_lift_{0}.csv'.format(split)
    if p.is_file():
        b = pd.read_csv(p)
        ax.plot(b['bin'], b['cum_lift'], marker='o', label=split)
ax.axhline(1.0, color='gray', lw=0.8, ls='--')
ax.set_xlabel('decile (1 = highest score)'); ax.set_ylabel('cumulative lift')
ax.set_title('cumulative lift by depth'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

In [ ]:
# 落盘的图
for name in ['roc_pr.png', 'ks.png', 'gain.png', 'lift_decile.png', 'calibration.png',
             'importance_gain.png', 'shap_bar.png', 'shap_beeswarm.png']:
    p = rd / 'plots' / name
    if p.is_file():
        print(name)
        display(Image(str(p)))

In [ ]:
# xc 5 模型横向汇总(同一 RUN_ID)。注意:各模型人群/标签不同(响应=全量 finish、
# 资质=finish 人群授信、e2e=全量授信),指标不能跨行直接比;有意义的对比是
# 同类(资质 V1 vs V2、e2e V1 vs V2)与同产品跨 run。两段式 vs e2e 的真正对照
# 在融合漏斗(run_funnel_eval.py / 06)里看
XC_PRODUCTS = ['xc_resp_finish', 'xc_qual_finish', 'xc_qual_finish_1v1',
               'xc_e2e_credit', 'xc_e2e_credit_1v1']
rows = []
for prod in XC_PRODUCTS:
    try:
        prd = model_run_dir(load_config(prod), RUN_ID)
    except Exception:
        continue
    f = prd / 'metrics.json'
    if not f.is_file():
        continue
    mm = pd.read_json(str(f)).set_index('split')
    manp = prd / 'run_manifest.json'
    label = (json.loads(manp.read_text(encoding='utf-8')).get('label_column', '')
             if manp.is_file() else '')
    for split in ['valid', 'oot']:
        if split in mm.index:
            r = mm.loc[split]
            rows.append({'product': prod, 'label': label, 'split': split, 'n': int(r['n']),
                         'base_rate': r['base_rate'], 'pr_auc': r['pr_auc'], 'ks': r['ks'],
                         'precision_at_k': r['precision_at_k'], 'lift_at_k': r['lift_at_k']})
if rows:
    display(pd.DataFrame(rows).set_index(['product', 'split']).round(4))
else:
    print('未找到任何 xc 产品 RUN_ID={0} 的 metrics.json'.format(RUN_ID))